# Практична робота № 3


## Крок 1. Підготовка середовища та завантаження даних


In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive   # монтування Google Диска
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PATH = '/content/drive/MyDrive/sales.csv'

# Якщо файлу sales.csv немає на Диску - створюємо навчальний набір про продажі кави
import os
if not os.path.exists(PATH):
    rng = np.random.default_rng(42)
    n = 300
    drinks = {"Latte": 38.7, "Americano": 28.9, "Cappuccino": 32.0, "Espresso": 25.0, "Hot Chocolate": 35.8, "Cocoa": 35.8}
    names = rng.choice(list(drinks), n)
    cash = rng.choice(["card", "cash"], n, p=[0.9, 0.1])
    dt = pd.to_datetime("2024-03-01 08:00") + pd.to_timedelta(np.sort(rng.integers(0, 60*24*60, n)), unit="m")
    pd.DataFrame({
        "date": dt.date,
        "datetime": dt,
        "cash_type": cash,
        "card": [f"ANON-0000-0000-{i:04d}" if c == "card" else np.nan for i, c in enumerate(cash, 1)],
        "money": [round(drinks[x] * rng.choice([1.0, 1.0, 1.0, 1.1]), 2) for x in names],
        "coffee_name": names,
    }).to_csv(PATH, index=False)
    print("Створено навчальний файл:", PATH)

df = pd.read_csv(PATH)
df.head()

,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 08:00:00,card,ANON-0000-0000-0000,24.0,Espresso
1,2024-03-01,2024-03-01 08:47:00,cash,ANON-0000-0000-0001,35.0,Cocoa
2,2024-03-01,2024-03-01 09:34:00,cash,ANON-0000-0000-0002,32.0,Cappuccino
3,2024-03-01,2024-03-01 10:21:00,card,ANON-0000-0000-0003,35.0,Cocoa
4,2024-03-01,2024-03-01 11:08:00,card,ANON-0000-0000-0004,35.0,Cocoa


## Крок 2. Виявлення та обробка пропущених значень


In [4]:
data_messy = {
    'name': ['Anna', None, 'Bohdan', 'Olena', None],
    'age': [20, 21, None, 22, 24],
    'score': [85.5, 90.0, 78.3, None, 92.1]
}
df_messy = pd.DataFrame(data_messy)
df_messy

,name,age,score
0,Anna,20.0,85.5
1,None,21.0,90.0
2,Bohdan,NaN,78.3
3,Olena,22.0,NaN
4,None,24.0,92.1


In [5]:
print(df_messy.isnull().sum())   # кількість пропусків у кожній колонці

name     2
age      1
score    1
dtype: int64


In [6]:
# Видалення рядків з пропусками у колонці 'name'
df_dropped = df_messy.dropna(subset=['name'])
print(df_dropped)

# Заповнення пропусків у колонці 'age' середнім значенням
mean_age = df_messy['age'].mean()          # середнє, ігноруючи NaN
df_filled = df_messy.copy()
df_filled['age'] = df_filled['age'].fillna(mean_age)
print(df_filled)

     name   age  score
0    Anna  20.0   85.5
2  Bohdan   NaN   78.3
3   Olena  22.0    NaN
     name    age  score
0    Anna  20.00   85.5
1    None  21.00   90.0
2  Bohdan  21.75   78.3
3   Olena  22.00    NaN
4    None  24.00   92.1


## Крок 3. Виявлення та видалення дублікатів

In [7]:
data_dup = {
    'id': [1, 2, 3, 2, 3],
    'value': ['A', 'B', 'C', 'B', 'C']
}
df_dup = pd.DataFrame(data_dup)
df_dup

,id,value
0,1,A
1,2,B
2,3,C
3,2,B
4,3,C


In [8]:
print(df_dup.duplicated())
print("Кількість дублікатів:", df_dup.duplicated().sum())

0    False
1    False
2    False
3     True
4     True
dtype: bool
Кількість дублікатів: 2


In [9]:
df_unique = df_dup.drop_duplicates()   # залишає перше входження кожного рядка
print(df_unique)

   id value
0   1     A
1   2     B
2   3     C


## Крок 4. Виявлення та обробка викидів (метод IQR)

In [10]:
Q1 = df['money'].quantile(0.25)
Q3 = df['money'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1 = {Q1}, Q3 = {Q3}, IQR = {IQR}")
print(f"Нижня межа: {lower}, Верхня межа: {upper}")

Q1 = 28.9, Q3 = 35.0, IQR = 6.100000000000001
Нижня межа: 19.749999999999996, Верхня межа: 44.150000000000006


In [11]:
outliers = df[(df['money'] < lower) | (df['money'] > upper)]
print(f"Знайдено викидів: {len(outliers)}")

Знайдено викидів: 0


## Крок 5. Перетворення типів даних

In [12]:
print(df.dtypes, "\n")                                            # типи до перетворення
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')  # текст -> datetime64
print(df.dtypes)                                                  # типи після перетворення

date            object
datetime        object
cash_type       object
card            object
money          float64
coffee_name     object
dtype: object 

date                   object
datetime       datetime64[ns]
cash_type              object
card                   object
money                 float64
coffee_name            object
dtype: object


## Крок 6. Обробка рядкових даних

In [13]:
df_text = pd.DataFrame({
    'city': [' Kyiv ', 'LVIV', 'Odesa ', 'KHARKIV', ' kyiv '],
    'phone': ['(044) 123-45-67', '(032) 234-56-78', '(048) 345-67-89', '(057) 456-78-90', '(044) 123-45-67']
})
df_text

,city,phone
0,Kyiv,(044) 123-45-67
1,LVIV,(032) 234-56-78
2,Odesa,(048) 345-67-89
3,KHARKIV,(057) 456-78-90
4,kyiv,(044) 123-45-67


In [14]:
df_text['city'] = df_text['city'].str.strip().str.lower()                        # без пробілів, нижній регістр
df_text['phone'] = df_text['phone'].str.replace(r'\(|\)|-', '', regex=True)        # без дужок і дефісів
print(df_text)

      city        phone
0     kyiv  044 1234567
1     lviv  032 2345678
2    odesa  048 3456789
3  kharkiv  057 4567890
4     kyiv  044 1234567


## Крок 7. Створення нових ознак (feature engineering)

In [15]:
df['date_only'] = df['datetime'].dt.date
df['hour'] = df['datetime'].dt.hour
df['day_of_week'] = df['datetime'].dt.dayofweek   # 0 - понеділок, 6 - неділя
df['month'] = df['datetime'].dt.month
print(df[['datetime', 'date_only', 'hour', 'day_of_week', 'month']].head())

             datetime   date_only  hour  day_of_week  month
0 2024-03-01 08:00:00  2024-03-01     8            4      3
1 2024-03-01 08:47:00  2024-03-01     8            4      3
2 2024-03-01 09:34:00  2024-03-01     9            4      3
3 2024-03-01 10:21:00  2024-03-01    10            4      3
4 2024-03-01 11:08:00  2024-03-01    11            4      3


## Крок 8. Перетворення категоріальних змінних (one-hot encoding)

In [16]:
df_encoded = pd.get_dummies(df, columns=['coffee_name'])
print(df_encoded.head())

         date            datetime cash_type                 card  money  \
0  2024-03-01 2024-03-01 08:00:00      card  ANON-0000-0000-0000   24.0   
1  2024-03-01 2024-03-01 08:47:00      cash  ANON-0000-0000-0001   35.0   
2  2024-03-01 2024-03-01 09:34:00      cash  ANON-0000-0000-0002   32.0   
3  2024-03-01 2024-03-01 10:21:00      card  ANON-0000-0000-0003   35.0   
4  2024-03-01 2024-03-01 11:08:00      card  ANON-0000-0000-0004   35.0   

    date_only  hour  day_of_week  month  coffee_name_Americano  \
0  2024-03-01     8            4      3                  False   
1  2024-03-01     8            4      3                  False   
2  2024-03-01     9            4      3                  False   
3  2024-03-01    10            4      3                  False   
4  2024-03-01    11            4      3                  False   

   coffee_name_Cappuccino  coffee_name_Cocoa  coffee_name_Espresso  \
0                   False              False                  True   
1           

## Крок 9. Нормалізація та стандартизація

In [17]:
df['money_norm'] = (df['money'] - df['money'].min()) / (df['money'].max() - df['money'].min())  # діапазон [0, 1]
df['money_std'] = (df['money'] - df['money'].mean()) / df['money'].std()                          # середнє 0, std 1

In [18]:
print(df[['money', 'money_norm', 'money_std']].head())
print("Мінімум money_norm:", df['money_norm'].min())
print("Максимум money_norm:", df['money_norm'].max())
print("Середнє money_std:", df['money_std'].mean())
print("Стандартне відхилення money_std:", df['money_std'].std())

   money  money_norm  money_std
0   24.0    0.000000  -1.540419
1   35.0    0.748299   0.667156
2   32.0    0.544218   0.065090
3   35.0    0.748299   0.667156
4   35.0    0.748299   0.667156
Мінімум money_norm: 0.0
Максимум money_norm: 1.0
Середнє money_std: -2.1908401019269756e-16
Стандартне відхилення money_std: 0.9999999999999989


## Крок 10. Збереження очищених даних

In [19]:
final_df = df[['datetime', 'cash_type', 'money', 'coffee_name', 'hour', 'day_of_week', 'month', 'money_norm', 'money_std']]
final_df.to_csv('/content/drive/MyDrive/cleaned_sales.csv', index=False)

In [20]:
final_df.head()

,datetime,cash_type,money,coffee_name,hour,day_of_week,month,money_norm,money_std
0,2024-03-01 08:00:00,card,24.0,Espresso,8,4,3,0.000000,-1.540419
1,2024-03-01 08:47:00,cash,35.0,Cocoa,8,4,3,0.748299,0.667156
2,2024-03-01 09:34:00,cash,32.0,Cappuccino,9,4,3,0.544218,0.065090
3,2024-03-01 10:21:00,card,35.0,Cocoa,10,4,3,0.748299,0.667156
4,2024-03-01 11:08:00,card,35.0,Cocoa,11,4,3,0.748299,0.667156
